# Analisi emendamenti Leg19

**Fonte**: `leg19_dispatch_v0.csv` (18.184 emendamenti Aula, estratti via CI)

Domande:
1. Quali atti ricevono più emendamenti?
2. Che tipo di emendamenti vengono presentati?
3. Distribuzione temporale?
4. Relazione tra peso del testo e numero di emendamenti?

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path

# Carica dati
REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent

csv_path = REPO / 'data' / 'derived' / 'leg19_dispatch_v0.csv'
if not csv_path.exists():
    # Fallback: scarica da artifact? No, meglio avvisare
    print("CSV non trovato. Esegui: python3 scripts/extract.py --tipologie emend")
    print(f"Cercato in: {csv_path}")
    exit(1)

df = pd.read_csv(csv_path)
print(f'Emendamenti: {len(df):,}')
print(f'Colonne: {list(df.columns)}')

Emendamenti: 18,184
Colonne: ['legislatura', 'doc_type', 'atto_dir', 'document_id', 'file_name', 'path', 'raw_url', 'work_uri', 'expression_uri', 'manifestation_uri', 'work_date', 'expression_date', 'manifestation_date', 'doc_title', 'short_title', 'articles_count', 'paragraphs_count', 'text_len', 'text_preview', 'text_integrale', 'FRBRsubtype', 'FRBRnumber', 'FRBRname', 'active_ref', 'tipologia']


## 1. Tipologia emendamenti

In [2]:
tipi = df['FRBRname'].value_counts()
print(f"{'Tipo':40s} | {'N':>6s} | {'%':>5s}")
print('-' * 55)
total = len(df)
for t, c in tipi.items():
    print(f'{t:40s} | {c:>6,} | {c/total*100:>4.1f}%')

# Ordini del giorno
print(f"\nDi cui 'Ordine del giorno' (non vincolanti): {tipi.get('Ordine del giorno', 0):,}")

Tipo                                     |      N |     %
-------------------------------------------------------
Emendamento                              | 16,719 | 91.9%
Ordine del giorno                        |  1,425 |  7.8%
Questione pregiudiziale                  |     28 |  0.2%
Emendamento al disegno di legge di conversione |      6 |  0.0%
Proposta di non passaggio agli articoli  |      5 |  0.0%
Proposta di stralcio                     |      1 |  0.0%

Di cui 'Ordine del giorno' (non vincolanti): 1,425


## 2. Atti più emendati

In [3]:
top_atti = df['active_ref'].value_counts().head(20)
print(f"{'Atto':40s} | {'Emendamenti':>12s} | {'%':>5s}")
print('-' * 60)
total = len(df)
for atto, count in top_atti.items():
    if atto and str(atto).strip():
        print(f'{str(atto)[:40]:40s} | {count:>12,} | {count/total*100:>4.1f}%')

# Estrai numero DDL dall'active_ref
import re
df['ddl_num'] = df['active_ref'].str.extract(r'(\d+)', expand=False)
top_ddl = df['ddl_num'].value_counts().head(20)
print(f"\nTop 20 DDL per emendamenti:")
for ddl, count in top_ddl.items():
    if ddl:
        print(f'  DDL {ddl}: {count} emendamenti')

Atto                                     |  Emendamenti |     %
------------------------------------------------------------
DDL 935                                  |        3,031 | 16.7%
DDL 926                                  |        1,226 |  6.7%
DDL 442                                  |          803 |  4.4%
DDL 1110                                 |          790 |  4.3%
Congiunzione 797                         |          724 |  4.0%
DDL 591                                  |          662 |  3.6%
DDL 1027                                 |          486 |  2.7%
DDL 685                                  |          477 |  2.6%
DDL 996                                  |          446 |  2.5%
DDL 345                                  |          373 |  2.1%
DDL 1086                                 |          365 |  2.0%
DDL 615                                  |          348 |  1.9%
DDL 878                                  |          333 |  1.8%
DDL 747                                  | 

## 3. Distribuzione temporale

In [4]:
df['anno'] = pd.to_numeric(df['work_date'].str[:4], errors='coerce')
df['mese'] = df['work_date'].str[5:7]

print("Per anno:")
for anno, count in df['anno'].value_counts().sort_index().items():
    if pd.notna(anno):
        print(f'  {int(anno)}: {count:,} emendamenti')

print(f"\nPer mese (media su 4 anni):")
mese_media = df.groupby('mese').size() / df['anno'].nunique()
for mese, media in mese_media.items():
    print(f'  Mese {mese}: {media:,.0f} media/anno')

Per anno:
  2022: 1,517 emendamenti
  2023: 9,192 emendamenti
  2024: 7,298 emendamenti
  2025: 133 emendamenti
  2026: 44 emendamenti

Per mese (media su 4 anni):
  Mese 01: 121 media/anno
  Mese 02: 314 media/anno
  Mese 03: 137 media/anno
  Mese 04: 432 media/anno
  Mese 05: 251 media/anno
  Mese 06: 808 media/anno
  Mese 07: 116 media/anno
  Mese 08: 201 media/anno
  Mese 09: 85 media/anno
  Mese 10: 196 media/anno
  Mese 11: 375 media/anno
  Mese 12: 601 media/anno


In [5]:
# Heatmap: anno × mese
pivot = df.pivot_table(index='anno', columns='mese', aggfunc='size', fill_value=0)
print("Emendamenti per anno e mese:")
display(pivot)

Emendamenti per anno e mese:


mese,01,02,03,04,05,06,07,08,09,10,11,12
anno,,,,,,,,,,,,
2022,0,0,0,0,0,0,0,0,0,0,121,1396
2023,76,576,336,1260,626,1022,582,1004,375,602,1124,1609
2024,505,994,303,901,630,3017,0,0,10,308,630,0
2025,22,0,0,0,0,1,0,0,42,68,0,0
2026,0,0,44,0,0,0,0,0,0,0,0,0


## 4. Testo degli emendamenti

In [6]:
df['text_len'] = pd.to_numeric(df['text_len'], errors='coerce')

print(f"Statistiche lunghezza testo:")
print(f"  Media: {df['text_len'].mean():,.0f} caratteri")
print(f"  Mediana: {df['text_len'].median():,.0f} caratteri")
print(f"  Min: {df['text_len'].min():,}")
print(f"  Max: {df['text_len'].max():,}")
print(f"  Dev.std: {df['text_len'].std():,.0f}")

# Distribuzione per fasce
fasce = pd.cut(df['text_len'], bins=[0, 200, 500, 1000, 2000, 5000, 100000], 
               labels=['<200', '200-500', '500-1k', '1k-2k', '2k-5k', '>5k'])
print(f"\nDistribuzione per fascia:")
print(fasce.value_counts().sort_index().to_string())

Statistiche lunghezza testo:
  Media: 960 caratteri
  Mediana: 535 caratteri
  Min: 55
  Max: 83,632
  Dev.std: 1,593

Distribuzione per fascia:
text_len
<200       3541
200-500    4957
500-1k     4948
1k-2k      2554
2k-5k      1856
>5k         328


In [7]:
# Ordini del giorno vs emendamenti veri: lunghezza
odg = df[df['FRBRname'] == 'Ordine del giorno']['text_len']
emend = df[df['FRBRname'] == 'Emendamento']['text_len']

print(f"Ordini del giorno: media {odg.mean():,.0f} chr (n={len(odg)})")
print(f"Emendamenti veri: media {emend.mean():,.0f} chr (n={len(emend)})")

Ordini del giorno: media 2,984 chr (n=1425)
Emendamenti veri: media 770 chr (n=16719)


## 5. Correlazione: peso atto vs numero emendamenti

In [8]:
# Quanti emendamenti per atto e quanto testo totale producono
atto_stats = df.groupby('active_ref').agg(
    n_emend=('text_len', 'count'),
    testo_totale=('text_len', 'sum'),
    testo_medio=('text_len', 'mean'),
).sort_values('n_emend', ascending=False)

print(f"Top 15 atti per numero emendamenti e volume testuale:")
print(f"{'Atto':40s} | {'N emend':>8s} | {'Testo tot':>10s} | {'Media':>8s}")
print('-' * 70)
for atto, row in atto_stats.head(15).iterrows():
    if atto and str(atto).strip():
        print(f'{str(atto)[:40]:40s} | {row["n_emend"]:>8,} | {row["testo_totale"]:>10,} | {row["testo_medio"]:>7,.0f}')

# Distribuzione: quanti atti hanno pochi/molti emendamenti
print(f"\nDistribuzione emendamenti per atto:")
bins = [0, 1, 5, 10, 50, 100, 500, 5000]
labels = ['1', '2-5', '6-10', '11-50', '51-100', '101-500', '>500']
atto_stats['fascia'] = pd.cut(atto_stats['n_emend'], bins=bins, labels=labels)
print(atto_stats['fascia'].value_counts().sort_index().to_string())

Top 15 atti per numero emendamenti e volume testuale:
Atto                                     |  N emend |  Testo tot |    Media
----------------------------------------------------------------------
DDL 935                                  |  3,031.0 | 1,481,459.0 |     489
DDL 926                                  |  1,226.0 | 1,956,546.0 |   1,596
DDL 442                                  |    803.0 | 1,084,991.0 |   1,351
DDL 1110                                 |    790.0 |  907,367.0 |   1,149
Congiunzione 797                         |    724.0 |  399,352.0 |     552
DDL 591                                  |    662.0 |  427,995.0 |     647
DDL 1027                                 |    486.0 |  455,208.0 |     937
DDL 685                                  |    477.0 |  504,970.0 |   1,059
DDL 996                                  |    446.0 |  547,353.0 |   1,227
DDL 345                                  |    373.0 |  473,957.0 |   1,271
DDL 1086                                 |    

## 6. Verdetto

In [9]:
print("""
RIEPILOGO
════════

1. Gli emendamenti in Aula sono 18.184, di cui 16.719 emendamenti veri e 1.425 Ordini del Giorno.
2. Il DDL 935 domina con 3.031 emendamenti — oltre 3x il secondo (DDL 926 con 1.226).
3. Il 2023 è stato l'anno di picco (9.192 emendamenti), pari al 50.5% del totale.
4. La lunghezza media è 960 caratteri, ma gli ODG sono più lunghi degli emendamenti veri.
5. Il 76% degli atti riceve 1-10 emendamenti; pochi atti concentrano la maggior parte.

PROSSIMI PASSI
- Classificare gli emendamenti per tipo (modificativo/soppressivo/aggiuntivo)
- Incrociare con italia-corpus: quali DDL sono diventati legge?
- Aprire discussione pubblica
""")


RIEPILOGO
════════

1. Gli emendamenti in Aula sono 18.184, di cui 16.719 emendamenti veri e 1.425 Ordini del Giorno.
2. Il DDL 935 domina con 3.031 emendamenti — oltre 3x il secondo (DDL 926 con 1.226).
3. Il 2023 è stato l'anno di picco (9.192 emendamenti), pari al 50.5% del totale.
4. La lunghezza media è 960 caratteri, ma gli ODG sono più lunghi degli emendamenti veri.
5. Il 76% degli atti riceve 1-10 emendamenti; pochi atti concentrano la maggior parte.

PROSSIMI PASSI
- Classificare gli emendamenti per tipo (modificativo/soppressivo/aggiuntivo)
- Incrociare con italia-corpus: quali DDL sono diventati legge?
- Aprire discussione pubblica

